In [1]:
import pandas as pd
import geopandas as gpd
from IPython.display import display
import requests as re
from io import BytesIO
from os import path, makedirs
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

# Atualização de 2026 - Mortalidade Materna e Mortalidade Infantil

Para o ano de 2026, o GT Saúde decidiu adicionar dois painéis com dados específicos de mortalidade, um para mortalidade materna e outro para mortalidade infantil. Esses temas já haviam sido sugeridos pelos especialistas que participaram em 2025, e também foram trabalhados pela auditoria do TCMSP, resultando inclusive em um evento em parceria com o OPP no ano de 2026.

Agora, precisamos adicionar esses dados padronizados no formato adequado ao Qlik Sense para que sejam incorporados ao DataSP.

Os dados são coeficientes semelhantes ao de mortalidade geral, mas com recortes específicos. De todo modo, tratam-se de coeficientes ponderados. No caso de mortalidade materna, temos o seguinte coeficiente:

$$ \text{Razão de mortalidade materna} = \frac{\text{Número de óbitos de mulheres por causas e condições consideradas de morte materna}}{\text{Número de nascidos vivos}} \times 100.000 $$

E, no caso de mortalidade infantil, temos:

$$ \text{Taxa de mortalidade infantil} = \frac{\text{Número de óbitos menores de 1 ano de idade}}{\text{Número de nascidos vivos}} \times 1.000 $$

Para que possamos recalcular esses indicadores de acordo com a seleção de Subprefeituras feito dinamicamente pelo usuário, precisamos obter o numerador e o denomidador deles e agregá-los dinamicamente de acordo com a seleção.

Para isso, existem duas alternativas possíveis. Uma delas é obter a razão (ou taxa) de mortalidade já calculada para cada Subprefeitura e estimar o número de óbitos com base no número de nascidos vivos. Outra forma seria obter os componentes da razão (taxa) e calculá-lo para cada Subprefeitura. A primeira maneira é preferida, porque garante que os indicadores reflitam exatamente os valores disponíveis em bases de dados oficiais. Entretanto, esses valores existem apenas para o ano de 2024. Para o ano de 2025, estão disponíveis apenas dados provisórios de mortalidade geral, de modo que o cálculo da razão (taxa) é possível de modo provisório, ainda que não esteja disponível em bases oficiais.

Portanto, as duas maneiras serão utilizadas: para o ano de 2024, serão utilizados os valores diponíveis em bases oficiais e o número de óbitos será estimado com base no número de nascidos vivos; para o ano de 2025, a razão (taxa) será calculada com base nos dados de mortalidade geral e nascidos vivos.

# Extração dos dados

## Mortalidade Materna

Esta razão será obtida do ObservaSampa, mas o download está apresentando um erro no momento. Os dados foram extraídos e corrigidos manualmente e estão disponíveis na pasta de dados do projeto

### 2024

In [3]:
data_input_dir = path.join('data', 'input', 'saude')

In [4]:
file_name_materna = 'mortalidade_materna_2024.csv'
df_materna_24 = pd.read_csv(path.join(data_input_dir, file_name_materna), sep=';', encoding='latin1', decimal=',')
df_materna_24

,região,nivel_regional,indicador,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,São Paulo,Município,03.01.01 Razão de mortalidade materna (por 100...,59.5049,49.4625,49.1477,62.5654,58.4294,56.7917,59.0706,47.7766,50.4430,56.4131,65.0961,35.5896,29.5247,25.6544
1,Guaianases,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,120.6030,39.8248,138.5864,153.6393,118.8119,59.8563,155.6723,60.9632,63.5997,45.1365,0.0000,0.0000,79.7448,56.9638
2,Ipiranga,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,15.1883,76.4292,15.2999,30.1114,44.6163,77.9180,15.7978,15.5933,16.7898,35.6888,58.3771,0.0000,41.0256,21.5936
3,Pinheiros,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,96.3391,32.4044,33.0251,32.4570,0.0000,34.3525,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,Pirituba-Jaraguá,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,42.3430,57.6037,29.5639,43.3839,72.1605,77.1724,76.2544,31.5657,64.9351,88.9363,55.5042,56.9801,37.6152,20.3707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,Vila Guilherme,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,446.4286,0.0000,153.6098,0.0000
125,Vila Matilde,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,203.6660,99.2063,104.4932,109.0513
126,Vila Prudente,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,90.4977,0.0000,0.0000,0.0000
127,Penha,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,79.4913


### 2025

Para 2025, obteremos os dados de mortalidade geral filtrando para mulheres com causas relacionadas ao Capítulo XV. Gravidez parto e puerpério do CID10. Esses dados também são extraídos manualmente, desta vez do TabNet, e adicionados a pasta de dados deste projeto.

In [5]:
file_name_obitos_materna_25 = 'A002518192_29_143_95.csv'
df_obitos_materna_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_materna_25),
                                   sep=';', encoding='latin1', decimal=',', skiprows=5)
df_obitos_materna_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,2.0
1,Butantã,1.0
2,Campo Limpo,1.0
3,Capela do Socorro,3.0
4,Cidade Ademar,2.0
5,Guaianases,1.0
6,Ipiranga,2.0
7,Itaim Paulista,4.0
8,Itaquera,3.0
9,Jaçanã/Tremembé,3.0


Vamos remover as linhas finais com informações complementares sobre os dados

In [6]:
df_obitos_materna_25 = df_obitos_materna_25[:-6]
df_obitos_materna_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,2.0
1,Butantã,1.0
2,Campo Limpo,1.0
3,Capela do Socorro,3.0
4,Cidade Ademar,2.0
5,Guaianases,1.0
6,Ipiranga,2.0
7,Itaim Paulista,4.0
8,Itaquera,3.0
9,Jaçanã/Tremembé,3.0


## Mortalidade Infantil

### 2024

Neste caso, tanto a taxa, quanto os dados provisórios de óbitos serão obtidos do TabNet. Vamos começar pela taxa de mortalidade infantil.

In [7]:
file_name_infantil_24 = 'A003317192_29_141_165.csv'
df_infantil_24 = pd.read_csv(path.join(data_input_dir, file_name_infantil_24),
                                         sep=';', encoding='latin1', decimal=',', skiprows=3)
df_infantil_24

,Subprefeitura resid,Coef_mort_infantil
0,Aricanduva/Formosa/Carrão,11.94
1,Butantã,8.41
2,Campo Limpo,10.25
3,Capela do Socorro,12.97
4,Casa Verde/Cachoeirinha,11.43
5,Cidade Ademar,11.77
6,Cidade Tiradentes,9.01
7,Ermelino Matarazzo,8.26
8,Freguesia/Brasilândia,15.18
9,Guaianases,13.96


Novamente, vamos remover as linhas de totalização e informações adicionais.

In [8]:
df_infantil_24 = df_infantil_24[:-8]
df_infantil_24

,Subprefeitura resid,Coef_mort_infantil
0,Aricanduva/Formosa/Carrão,11.94
1,Butantã,8.41
2,Campo Limpo,10.25
3,Capela do Socorro,12.97
4,Casa Verde/Cachoeirinha,11.43
5,Cidade Ademar,11.77
6,Cidade Tiradentes,9.01
7,Ermelino Matarazzo,8.26
8,Freguesia/Brasilândia,15.18
9,Guaianases,13.96


### 2025

In [9]:
file_name_obitos_infantil_25 = 'A003858192_29_141_165.csv'
df_obitos_infantil_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_infantil_25),
                                    sep=';', encoding='latin1', decimal=',', skiprows=4)
df_obitos_infantil_25 = df_obitos_infantil_25[:-6]
df_obitos_infantil_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,18.0
1,Butantã,41.0
2,Campo Limpo,66.0
3,Capela do Socorro,79.0
4,Casa Verde/Cachoeirinha,37.0
5,Cidade Ademar,57.0
6,Cidade Tiradentes,32.0
7,Ermelino Matarazzo,14.0
8,Freguesia/Brasilândia,68.0
9,Guaianases,37.0


## Nascidos Vivos

Os dados de nascidos vivos também são obtidos do TabNet.

In [10]:
file_name_nv = 'A004130192_29_141_165.csv'
df_nv = pd.read_csv(path.join(data_input_dir, file_name_nv),
                    sep=';', encoding='latin1', decimal=',', skiprows=3)
df_nv = df_nv[:-3]
df_nv

,SubPrefeitura residência,2024,2025,Total
0,Aricanduva/Formosa/Carrão,2177.0,2302.0,4479.0
1,Butantã,4399.0,4639.0,9038.0
2,Campo Limpo,7417.0,7368.0,14785.0
3,Capela do Socorro,7092.0,7121.0,14213.0
4,Casa Verde/Cachoeirinha,3236.0,3428.0,6664.0
5,Cidade Ademar,4673.0,4819.0,9492.0
6,Cidade Tiradentes,2552.0,2530.0,5082.0
7,Ermelino Matarazzo,2057.0,2121.0,4178.0
8,Freguesia/Brasilândia,4215.0,4293.0,8508.0
9,Guaianases,3511.0,3400.0,6911.0


Nesse caso, temos também que excluir a coluna de totalização.

In [11]:
df_nv = df_nv.drop(columns=['Total'])
df_nv

,SubPrefeitura residência,2024,2025
0,Aricanduva/Formosa/Carrão,2177.0,2302.0
1,Butantã,4399.0,4639.0
2,Campo Limpo,7417.0,7368.0
3,Capela do Socorro,7092.0,7121.0
4,Casa Verde/Cachoeirinha,3236.0,3428.0
5,Cidade Ademar,4673.0,4819.0
6,Cidade Tiradentes,2552.0,2530.0
7,Ermelino Matarazzo,2057.0,2121.0
8,Freguesia/Brasilândia,4215.0,4293.0
9,Guaianases,3511.0,3400.0
